---
title: "Czas i trendy"
---

Szereg czasowy powinien od razu pokazywać jednostkę czasu, punkt odniesienia i
to, czy patrzymy na wartości bezwzględne, zmianę procentową czy indeks.

In [ ]:
#| label: setup-05
library(tidyverse)
library(here)
library(janitor)
library(lubridate)

source(here("R", "theme_course.R"))
theme_set(theme_course())

## Wykres liniowy (line chart) prostego trendu

In [ ]:
#| label: fig-climate-line
#| fig-cap: "CO₂ i względna temperatura w czasie."
#| fig-alt: "Wykres liniowy pokazuje wzrost stężenia CO₂ w kolejnych latach oraz zmienność temperatury względnej."
climate <- readr::read_csv(here("datasets", "climate_change.csv"), show_col_types = FALSE) |>
  mutate(date = lubridate::ymd(date))

climate |>
  ggplot(aes(x = date, y = co2)) +
  geom_line(colour = "#0072B2", linewidth = 0.8) +
  labs(
    title = "Linia jest naturalnym wyborem dla ciągłej osi czasu",
    x = NULL,
    y = "Stężenie CO₂",
    caption = "Źródło: datasets/climate_change.csv"
  )

## Porównanie serii przez indeks

Gdy serie mają różne poziomy, często lepiej pokazać indeks od wspólnego punktu
startu niż surową cenę.

In [ ]:
#| label: fig-stock-index
#| fig-cap: "Indeks cen zamknięcia dla wybranych spółek."
#| fig-alt: "Wykres liniowy porównuje względną zmianę cen akcji AAPL, AMZN i MSFT od pierwszego dnia obserwacji."
stocks <- readr::read_csv(here("datasets", "stocks_cleaned.csv"), show_col_types = FALSE) |>
  janitor::clean_names() |>
  select(-any_of("x1")) |>
  mutate(date = lubridate::ymd(date))

stocks |>
  filter(name %in% c("AAPL", "AMZN", "MSFT")) |>
  arrange(name, date) |>
  group_by(name) |>
  mutate(close_index = close / first(close) * 100) |>
  ungroup() |>
  ggplot(aes(x = date, y = close_index, colour = name)) +
  geom_line(linewidth = 0.75) +
  scale_colour_course_d(name = "Spółka") +
  labs(
    title = "Indeks 100 ułatwia porównanie serii o różnych poziomach",
    x = NULL,
    y = "Cena zamknięcia, indeks = 100",
    caption = "Źródło: datasets/stocks_cleaned.csv"
  )

## Agregacja przed wykresem liniowym

In [ ]:
#| label: fig-bike-month
#| fig-cap: "Średnia dzienna liczba wypożyczeń rowerów według miesiąca."
#| fig-alt: "Linia miesięczna pokazuje sezonowość wypożyczeń rowerów z wyższymi wartościami w cieplejszych miesiącach."
bike_month <- readr::read_csv(here("datasets", "bike_share.csv"), show_col_types = FALSE) |>
  janitor::clean_names() |>
  mutate(date = lubridate::ymd(dteday), month = floor_date(date, "month")) |>
  group_by(month) |>
  summarise(avg_rentals = mean(total_rentals), .groups = "drop")

bike_month |>
  ggplot(aes(x = month, y = avg_rentals)) +
  geom_line(colour = "#009E73", linewidth = 0.9) +
  geom_point(colour = "#009E73", size = 1.7) +
  labs(
    title = "Agregacja powinna wynikać z pytania, nie z wygody",
    x = NULL,
    y = "Średnia dzienna liczba wypożyczeń",
    caption = "Źródło: datasets/bike_share.csv"
  )

## Zadanie

Dla `stocks_cleaned.csv` wybierz jedną spółkę i pokaż cenę zamknięcia. Następnie
agreguj dane do miesięcznej średniej i porównaj, co zyskujesz, a co tracisz.